# YHCT FAST5 Balanced — Judge có context + JSON đầu ra điều kiện

**Thử nghiệm tối ưu tốc độ, chưa đo thực tế trên T4 ×2.** Giữ rubric, ngữ cảnh trước/sau và các ví dụ thuật ngữ quan trọng trong **input**. Với câu **≥97 điểm, không có lỗi và không bất định**, yêu cầu model trả JSON ngắn `[ID, SCORE]`. Những câu còn lại trả đầy đủ `[ID, ERRORS, SCORE, NEEDS_CONTEXT, SOURCE_UNCERTAIN]` để phục vụ Repair. Mỗi câu vẫn được chấm điểm riêng và lưu checkpoint riêng.

Sử dụng hai chế độ `all3` và `tiebreak`, batch tự chia nhỏ khi thất bại. **Prompt v2 mới có signature checkpoint riêng**; không nhầm kết quả với prompt FAST5 cũ. Khi bật `REUSE_LEGACY_JUDGE_CHECKPOINTS=True`, chỉ tái sử dụng các phiếu V2 dài đã xác thực và đánh dấu rõ nguồn.


## 1. Thiết lập môi trường và cache model

In [ ]:
# CELL 1: thiết lập máy, GPU và HF cache. Chạy trước các thư viện transformers.
import os, sys, subprocess
from pathlib import Path
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/_yhct_work_v2')
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
WORK.mkdir(parents=True, exist_ok=True)
CACHE = Path('/tmp/yhct_v2_models_hf_cache')
CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE)
os.environ['HF_HUB_CACHE'] = str(CACHE/'hub')
os.environ['HF_HUB_DISABLE_XET'] = '1'
# Nếu môi trường đã cài được các thư viện, có thể đặt INSTALL_DEPENDENCIES=False.
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
       'transformers>=5.6,<6','accelerate>=1.2','bitsandbytes>=0.45',
       'sentencepiece','safetensors','huggingface_hub'])
# HF_TOKEN qua Kaggle Secrets (không ghi token vào notebook).
# Secret thiếu vẫn chạy được các model công khai; model gated sẽ cần token.
USE_HF_TOKEN = True
if USE_HF_TOKEN and Path('/kaggle/input').exists():
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import login
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
        if hf_token:
            login(token=hf_token, add_to_git_credential=False)
            print('HF_TOKEN: authenticated')
        else:
            print('[WARN] HF_TOKEN rỗng; dùng truy cập public nếu có thể')
    except Exception as exc:
        print('[WARN] HF_TOKEN chưa truy cập được:', type(exc).__name__, str(exc)[:120])
print('INPUT_ROOT:',INPUT_ROOT,'WORK:',WORK,'MODEL_CACHE:',CACHE)


## 2. Chọn Original Judge hoặc Re-judge

In [ ]:
# CELL 2: CẤU HÌNH JUDGE
MODE = 'original'   # 'original' = Judge TSV gốc; 'rejudge' = Judge rejudge_inputs.tsv do Repair xuất
SOURCE_TSV_PATH = "/kaggle/input/datasets/vntrangphm/data-nlp/YHCT_FINAL_part2.tsv"  # VD: '/kaggle/input/yhct-data/YHCT_FINAL_part2.tsv'
LEGACY_JUDGE_ZIP_PATH = None   # Có thể thử 29 câu từ ZIP Judge cũ, KHÔNG tái dùng điểm cũ
REJUDGE_TSV_PATH = None       # VD: '/kaggle/input/repair-output/rejudge_inputs.tsv'
SHARD_ID = None                # Auto lấy từ TSV đầu hoặc source_shard ở rejudge_inputs.tsv
AUTO_CONTEXT_FROM_ALL_PARTS = True
CONTEXT_WINDOW = 1           # 1 câu trước + 1 câu sau (0,1,2)
SAME_SECTION_ONLY = True     # Không đưa câu ở section khác vào làm ngữ cảnh
MAX_CONTEXT_CHARS_PER_SIDE = 650
JUDGE_PROMPT_LANGUAGE = 'en'  # 'en' hoặc 'vi', dùng cùng rubric và schema
JUDGE_SCORE_THRESHOLD = 95.0 # Ngưỡng tham chiếu; voting còn dựa trên lỗi và đồng thuận
# all3: thử đủ ba model trên cùng tập câu (đây là lựa chọn bạn yêu cầu).
# tiebreak: 7B + 8B chấm mọi câu; 14B chỉ chấm khi hai model bất đồng.
JUDGE_EXECUTION_MODE = 'tiebreak'
# Gộp nhiều câu trong MỘT prompt (không phải CUDA tensor batch).
# Batch lớn 8/10/6 tự chia nhỏ khi prompt vượt context, JSON lỗi hoặc OOM.
JUDGE_BATCH_SIZE = 8          # Fallback/default nếu tên model không có trong mapping
JUDGE_MODELS = [
    {'name':'Qwen/Qwen2.5-7B-Instruct','quant4':True},
    {'name':'Qwen/Qwen3-8B','quant4':True},
    {'name':'Qwen/Qwen3-14B','quant4':True},
]
JUDGE_BATCH_SIZES = {
    'Qwen/Qwen2.5-7B-Instruct': 8,
    'Qwen/Qwen3-8B': 10,
    'Qwen/Qwen3-14B': 6,
}
TEST_MAX_UNITS = None           # Full shard; đổi thành 200 để kiểm tra 200 câu đầu
MAX_NEW_INFERENCES = None     # None = không giới hạn số lượt, thời gian vẫn giới hạn
RUN_HOURS = 12.0               # Dừng mềm sau ~7 giờ kể từ khi bắt đầu CELL 7
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 180        # Giới hạn MỖI CÂU; batch tự nhân với JUDGE_BATCH_SIZE
MAX_CONSECUTIVE_ERRORS = 6
BATCH_BACKOFF_AFTER = 2    # Nếu batch lớn thất bại 2 nhóm, tự giảm kích thước cho model đó
MAX_JUDGE_ATTEMPTS = 3  # fallback mặc định
# Riêng 14B giảm thử lại từ 3 xuống 2 lần khi 1 câu JSON không hợp lệ.
# Thiếu source_span của omission được lưu để REVIEW, không retry.
JUDGE_ATTEMPTS_BY_MODEL = {
    'Qwen/Qwen2.5-7B-Instruct': 3,
    'Qwen/Qwen3-8B': 3,
    'Qwen/Qwen3-14B': 2,
}
UNTRANSLATED_HAN_ALLOWLIST = []  # Chỉ thêm cụm được nhóm duyệt, không tự xóa chữ Hán
VALID_JUDGE_BATCH_SIZES = set(range(1,13))
if JUDGE_BATCH_SIZE not in VALID_JUDGE_BATCH_SIZES:
    raise ValueError('JUDGE_BATCH_SIZE phải thuộc số nguyên 1–12')
model_names = {m['name'] for m in JUDGE_MODELS}
unknown_batch_keys = set(JUDGE_BATCH_SIZES) - model_names
if unknown_batch_keys:
    raise ValueError('Tên model không có trong JUDGE_MODELS: ' + str(sorted(unknown_batch_keys)))
for model_name, batch_size in JUDGE_BATCH_SIZES.items():
    if not isinstance(batch_size,int) or batch_size not in VALID_JUDGE_BATCH_SIZES:
        raise ValueError(f'Batch không hợp lệ {model_name}={batch_size}; chọn số nguyên 1–12')
EFFECTIVE_JUDGE_BATCH_SIZES = {
    m['name']: JUDGE_BATCH_SIZES.get(m['name'], JUDGE_BATCH_SIZE)
    for m in JUDGE_MODELS
}
if JUDGE_EXECUTION_MODE not in ('all3','tiebreak'):
    raise ValueError('JUDGE_EXECUTION_MODE phải là all3 hoặc tiebreak')
print('Judge:', MODE, '| execution:', JUDGE_EXECUTION_MODE, '| language:', JUDGE_PROMPT_LANGUAGE,
      '| batch per model:', EFFECTIVE_JUDGE_BATCH_SIZES)

MODULE_DIR = None  # Đặt '/kaggle/input/<ten-dataset-code>' nếu có nhiều bản .py V2.

# FAST5 BALANCED: giữ rubric và ngữ cảnh; chỉ rút JSON nếu SCORE >=97,
# errors=[] và không có cờ bất định. Các câu có lỗi vẫn trả đủ bằng chứng.
# MAX_NEW_TOKENS=180 là NGÂN SÁCH tối đa / câu (không phải độ dài bắt buộc).

# Explicitly migrate earlier LONG-PROMPT 7B checkpoints by verifying each row.
# They remain labeled as legacy; set False for a clean A/B comparison.
REUSE_LEGACY_JUDGE_CHECKPOINTS = True
# Import FAST5 checkpoint từ prompt/signature cũ khi hash từng câu và model khớp.
# Kết quả được đánh dấu cross-prompt migration; False nếu cần benchmark thuần nhất.
REUSE_PRIOR_FAST5_CHECKPOINTS = True
# Lần đầu nên bật True để Save Version CHỈ KIỂM TRA CHECKPOINT, không tải GPU.
# Sau khi kiểm tra đủ Input, đổi False để tiếp tục chấm trên T4 x2.
CHECKPOINT_AUDIT_ONLY = False


## 3. Import các tiện ích từ `yhct_v2_core.py`

Add Input bộ ba `.py` V2 vào Kaggle trước khi Run All. Nếu Kaggle tìm nhiều bản, cấu hình `MODULE_DIR` ở Cell 2.


In [ ]:
# CELL 3: import module V2 dùng chung thay vì copy hàm dài vào notebook.
# Để cả 3 file .py trong CÙNG một thư mục Kaggle Dataset/Add Input,
# hoặc đặt MODULE_DIR trỏ đến thư mục chứa chúng.
import sys, importlib, json
from pathlib import Path
MODULE_FILES = ('yhct_v2_core.py', 'yhct_judge_v2.py', 'yhct_repair_v2.py', 'yhct_judge_fast5.py')

def locate_module_dir(explicit=None):
    if explicit:
        candidate=Path(explicit).expanduser().resolve()
        if not candidate.is_dir() or not all((candidate / n).is_file() for n in MODULE_FILES):
            raise FileNotFoundError(f'MODULE_DIR phải chứa đủ {MODULE_FILES}: {candidate}')
        return candidate
    roots=[Path('/kaggle/working'), INPUT_ROOT, Path.cwd(), Path('/mnt/data')]
    candidates=set()
    for root in roots:
        if not root.is_dir():
            continue
        if all((root/n).is_file() for n in MODULE_FILES):
            candidates.add(root.resolve())
        for p in root.rglob('yhct_v2_core.py'):
            d=p.parent.resolve()
            if all((d/n).is_file() for n in MODULE_FILES):
                candidates.add(d)
    if not candidates:
        raise FileNotFoundError('Không tìm thấy đủ 3 file .py V2. Upload thành Kaggle Dataset/Add Input và/hoặc đặt MODULE_DIR.')
    if len(candidates)>1:
        raise RuntimeError('Tìm thấy nhiều bản module V2, hãy đặt MODULE_DIR chính xác: '+str(sorted(map(str,candidates))))
    return next(iter(candidates))

RESOLVED_MODULE_DIR=locate_module_dir(MODULE_DIR)
for _module_name in ('yhct_v2_core','yhct_judge_v2','yhct_repair_v2'):
    _loaded=sys.modules.get(_module_name)
    if _loaded is not None and Path(getattr(_loaded,'__file__','/no-file')).resolve().parent != RESOLVED_MODULE_DIR:
        raise RuntimeError('Module V2 đã import từ thư mục khác; Restart Session rồi Run All.')
sys.path.insert(0,str(RESOLVED_MODULE_DIR)) if str(RESOLVED_MODULE_DIR) not in sys.path else None
importlib.invalidate_caches()
from yhct_v2_core import (
    VERSION, sha_bytes, stable_hash, utc_now, valid_shard, read_tsv, dump_tsv,
    atomic_string, write_json, append_jsonl, load_jsonl, norm_input_row,
    chinese_leftover, build_context, parse_json_object, CausalModel,
    find_output_manifests, restore_checkpoints,
)
from collections import Counter
print('YHCT V2 modules:',RESOLVED_MODULE_DIR,'| VERSION:',VERSION)


## 4. Import logic từ module chuyên biệt

`yhct_judge_v2.py` quản lý prompt, schema, validation, score, voting.


In [ ]:
# CELL 4: Prompt, MQM-lite schema, xử lý lỗi và voting (import từ yhct_judge_v2.py).
from yhct_judge_v2 import (
    JUDGE_SYSTEM_EN, JUDGE_SYSTEM_VI, JUDGE_USER_EN, JUDGE_USER_VI,
    JUDGE_GLM_CALIBRATION_EN, JUDGE_GLM_CALIBRATION_VI, TRIAGE_VERSION, APPROVED_TCM_TERMS,
    ERROR_CATEGORIES, SEVERITIES, prompt_messages, parse_judge, same_issue,
    aggregate_judgments, judge_with_retries, prompt_messages_batch,
    parse_judge_batch, judge_batch_with_fallback, arbitrate_two_judges,
)

# Override only inference and prompt formatting; voting/validation remain V2.
from yhct_judge_fast5 import (
    FAST_JUDGE_VERSION, SHORT_OUTPUT_MIN_SCORE, FAST_SYSTEM_EN, FAST_SYSTEM_VI,
    judge_batch_with_fallback as judge_batch_with_fallback, migrate_legacy_judge_checkpoints,
    migrate_prior_fast5_checkpoints,
    aggregate_judgments as aggregate_judgments,
    arbitrate_two_judges as arbitrate_two_judges,
)
# Dùng majority voting V2; chỉ thêm guard nếu một lỗi omission không có bằng chứng.
# Không dùng quy tắc chênh lệch điểm để REVIEW.
print('FAST JUDGE:', FAST_JUDGE_VERSION,
      '| short result when verified clean score >=', SHORT_OUTPUT_MIN_SCORE)
print('Triage: original V2 majority consensus (no score-spread override)')


## 5. Chọn nguồn dữ liệu và tạo context trước/sau

In [ ]:
# CELL 5: LOAD DATA + CONTEXT
import zipfile, json

def legacy_zip_audit(path:Path):
    with zipfile.ZipFile(path) as z:
        names=[n for n in z.namelist() if n.endswith('/all_original_audit.jsonl') or n=='all_original_audit.jsonl']
        if len(names)!=1:raise ValueError('ZIP cần đúng một all_original_audit.jsonl')
        data=z.read(names[0]); rows=[]
        for l in data.decode('utf-8-sig').splitlines():
            if not l.strip():continue
            r=json.loads(l)
            rows.append({'unit_id':r['unit_id'],'han':r['zh'],'viet':r['initial_vi'],
                         'source_line':r['unit_id'].split('-S')[1], 'section':'',
                         'model':r.get('model','legacy-CLC')})
        return data, rows

if MODE not in ('original','rejudge'):raise ValueError(MODE)
if len({m['name'] for m in JUDGE_MODELS}) != len(JUDGE_MODELS):
    raise ValueError('Trùng model Judge')
if MODE=='original':
    if SOURCE_TSV_PATH and LEGACY_JUDGE_ZIP_PATH:
        raise ValueError('Chỉ chọn SOURCE_TSV_PATH HOẶC LEGACY_JUDGE_ZIP_PATH')
    tsv=None; legacy=None
    if SOURCE_TSV_PATH:tsv=Path(SOURCE_TSV_PATH)
    elif LEGACY_JUDGE_ZIP_PATH:legacy=Path(LEGACY_JUDGE_ZIP_PATH)
    else:
        options=sorted(INPUT_ROOT.rglob('YHCT_FINAL_part*.tsv'))
        if len(options)==1:tsv=options[0]
        elif len(options)>1:
            raise RuntimeError('Nhiều TSV đầu vào; đặt SOURCE_TSV_PATH: '+str(options))
        else:
            zips=[]
            for p in INPUT_ROOT.rglob('*.zip'):
                try:
                    with zipfile.ZipFile(p) as z:
                        if any(x.endswith('all_original_audit.jsonl') for x in z.namelist()):zips.append(p)
                except (OSError,zipfile.BadZipFile):pass
            if len(zips)!=1:raise RuntimeError('Không tìm thấy đúng một TSV/legacy ZIP; hãy chọn đường dẫn thủ công')
            legacy=zips[0]
    if tsv:
        if not tsv.is_file():raise FileNotFoundError(tsv)
        source_bytes=tsv.read_bytes()
        source_rows=read_tsv(tsv)
        bad=[r.get('unit_id') for r in source_rows if r.get('status','DONE')!='DONE']
        if bad:raise ValueError(f'Có {len(bad)} câu chưa DONE; ví dụ {bad[:3]}')
        shard_id=SHARD_ID or tsv.stem
        print('Source TSV:',tsv)
    else:
        if not legacy.is_file():raise FileNotFoundError(legacy)
        source_bytes,source_rows=legacy_zip_audit(legacy)
        shard_id=SHARD_ID or legacy.stem
        print('Source legacy ZIP:',legacy)
    if not source_rows:raise RuntimeError('Input rỗng')
    extras=[]
    if AUTO_CONTEXT_FROM_ALL_PARTS:
        for p in sorted(INPUT_ROOT.rglob('YHCT_FINAL_part*.tsv')):
            if tsv and p.resolve()==tsv.resolve():continue
            extras.extend(read_tsv(p))
    rows=build_context(source_rows,extras,window=CONTEXT_WINDOW,
                same_section=SAME_SECTION_ONLY,max_side_chars=MAX_CONTEXT_CHARS_PER_SIDE)
    for r in rows:
        r['candidate_id']=stable_hash(['original',r['unit_id'],r['han'],r['viet'],r['source_model']])
        r['original_vi']=r['viet'];r['repair_model']='';r['mode']='original'
else:
    if REJUDGE_TSV_PATH: tsv=Path(REJUDGE_TSV_PATH)
    else:
        choices=sorted(INPUT_ROOT.rglob('rejudge_inputs.tsv'))
        if len(choices)!=1:
            raise RuntimeError('Cần đúng một rejudge_inputs.tsv; đặt REJUDGE_TSV_PATH: '+str(choices))
        tsv=choices[0]
    source_bytes=tsv.read_bytes()
    source_rows=read_tsv(tsv)
    if not source_rows:raise RuntimeError('rejudge_inputs.tsv rỗng. Hãy kiểm tra REJUDGE_EXPORT_CLEAN_ONLY ở Repair.')
    source_shards={r.get('source_shard') for r in source_rows if r.get('source_shard')}
    if len(source_shards)>1:raise ValueError('Không trộn nhiều shard trong một rejudge notebook')
    shard_id=SHARD_ID or (next(iter(source_shards)) if source_shards else tsv.parent.name)
    rows=[]
    for r in source_rows:
        rr=norm_input_row(r)
        rr.update({'candidate_id':r.get('candidate_id') or stable_hash(['rejudge',rr['unit_id'],rr['han'],rr['viet'],r.get('repair_model'),r.get('mode')]),
           'ctx_before_han':r.get('ctx_before_han',''),'ctx_after_han':r.get('ctx_after_han',''),
           'original_vi':r.get('original_vi',''), 'repair_model':r.get('repair_model',''),
           'mode':r.get('mode',''), 'source_model':r.get('repair_model','')})
        rows.append(rr)
    print('Rejudge TSV:',tsv)
SHARD = valid_shard(str(shard_id))
if TEST_MAX_UNITS is not None:
    if not isinstance(TEST_MAX_UNITS,int) or TEST_MAX_UNITS<1:raise ValueError('TEST_MAX_UNITS phải >=1')
    if MODE=='original': rows=rows[:TEST_MAX_UNITS]
    else:
        chosen=list(dict.fromkeys(r['unit_id'] for r in rows))[:TEST_MAX_UNITS]
        rows=[r for r in rows if r['unit_id'] in set(chosen)]
ids=[r['candidate_id'] for r in rows]
if len(ids)!=len(set(ids)):raise ValueError('candidate_id trùng; rejudge variants phải khác ID')
SOURCE_SHA=sha_bytes(source_bytes)
print('SHARD:',SHARD,'| candidate rows:',len(rows),'| contextual:',sum(bool(r.get('ctx_before_han') or r.get('ctx_after_han')) for r in rows))
print('Source sha:',SOURCE_SHA)
print('First:',rows[0]['unit_id'],rows[0]['han'][:100])


## 6. Tạo nhiệm vụ và khôi phục checkpoint

In [ ]:
# CELL 6: CHECKPOINT MANIFEST + TASKS (model-first)
PROMPT_SIGNATURE = stable_hash({'version':VERSION,'stage':MODE,
    'prompt_language':JUDGE_PROMPT_LANGUAGE,'fast_judge_version':FAST_JUDGE_VERSION,'short_output_min_score':SHORT_OUTPUT_MIN_SCORE,
    'fast_system_en':FAST_SYSTEM_EN, 'fast_system_vi':FAST_SYSTEM_VI,'system_en':JUDGE_SYSTEM_EN,
    'system_vi':JUDGE_SYSTEM_VI,'user_en':JUDGE_USER_EN,'user_vi':JUDGE_USER_VI,
    'glm_calibration_en':JUDGE_GLM_CALIBRATION_EN,
    'glm_calibration_vi':JUDGE_GLM_CALIBRATION_VI,'triage_policy':TRIAGE_VERSION,
    'approved_terms':APPROVED_TCM_TERMS,
    'validation':'span-v4-calibration-and-conservative-triage','allowlist':UNTRANSLATED_HAN_ALLOWLIST})
SIGNATURE=PROMPT_SIGNATURE
OUTPUT_DIR=WORK/'yhct_judge_fast5'/SHARD/MODE/SIGNATURE[:14]
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
MANIFEST={'schema':VERSION,'shard_id':SHARD,'stage':MODE,'signature':SIGNATURE,
          'source_sha256':SOURCE_SHA,'candidate_rows':len(rows),'models':JUDGE_MODELS,
          'prompt_language':JUDGE_PROMPT_LANGUAGE,'score_threshold':JUDGE_SCORE_THRESHOLD,'context_window':CONTEXT_WINDOW,
          'triage_version':TRIAGE_VERSION, 'fast_judge_version':FAST_JUDGE_VERSION,'short_output_min_score':SHORT_OUTPUT_MIN_SCORE,'output_format':'conditional-short-high-or-full','execution_mode':JUDGE_EXECUTION_MODE,
          'batch_size':JUDGE_BATCH_SIZE,
          'batch_size_by_model':EFFECTIVE_JUDGE_BATCH_SIZES,
          'input_path':str(tsv if MODE=='rejudge' else (tsv or legacy)),
          'note':'Context is Chinese-only and NEVER separately judged; score is LLM subjective rating.'}
write_json(OUTPUT_DIR/'judge_manifest.json',MANIFEST)
TASKS=[]
for cfg in JUDGE_MODELS:
    for row in rows:
        key=stable_hash([SIGNATURE,SHARD,cfg,row['candidate_id'],row['han'],row['viet'],
                         row.get('ctx_before_han',''),row.get('ctx_after_han','')])
        TASKS.append((cfg,row,key))
keys={k for _,_,k in TASKS}
older=find_output_manifests(INPUT_ROOT,'judge_manifest.json',SIGNATURE,SHARD,MODE)
completed=restore_checkpoints(OUTPUT_DIR,older,'judge_checkpoint.jsonl',SIGNATURE,keys)
# IMPORTANT: carry forward all restored rows into THIS run's output. Otherwise
# Kaggle Save Version of a fully resumed run would silently lose its checkpoint.
# Preserve every valid prior checkpoint in this shard/signature, even when testing
# only the first 80 rows. A later full-shard run can recover all 1,600 old results
# from the latest output dataset alone. Key hashes still verify each source row.
archive = {}
for folder in [*older, OUTPUT_DIR]:
    for rec in load_jsonl(folder / 'judge_checkpoint.jsonl'):
        key = rec.get('key')
        if rec.get('ok') is not True or rec.get('signature') != SIGNATURE or not isinstance(key,str):
            continue
        if key in archive and archive[key].get('result') != rec.get('result'):
            raise ValueError('Conflicting checkpoint for key: '+key)
        archive[key] = rec
archive.update(completed)
# Old FAST5 output from a different signature is NOT found by
# find_output_manifests(...). Import it only after validating the old hash of
# each individual sentence, its context, model configuration, and source SHA.
if MODE=='original' and REUSE_PRIOR_FAST5_CHECKPOINTS:
    fullrows_fast=build_context(source_rows, extras, window=CONTEXT_WINDOW,
        same_section=SAME_SECTION_ONLY, max_side_chars=MAX_CONTEXT_CHARS_PER_SIDE)
    for rr in fullrows_fast:
        rr['candidate_id']=stable_hash(['original',rr['unit_id'],rr['han'],rr['viet'],rr['source_model']])
    migrated_fast, fast_audit=migrate_prior_fast5_checkpoints(
        INPUT_ROOT, fullrows_fast, SHARD, MODE, SIGNATURE,
        JUDGE_MODELS, stable_hash, utc_now, source_sha=SOURCE_SHA)
    for key,rec in migrated_fast.items():
        if key not in archive:
            archive[key]=rec
        if key in keys and key not in completed:
            completed[key]=archive[key]
    print('Prior FAST5 cross-signature migration:',fast_audit)
else:
    print('Prior FAST5 cross-signature migration: disabled')
if MODE=='original' and REUSE_LEGACY_JUDGE_CHECKPOINTS:
    fullrows=build_context(source_rows, extras, window=CONTEXT_WINDOW,
        same_section=SAME_SECTION_ONLY, max_side_chars=MAX_CONTEXT_CHARS_PER_SIDE)
    for rr in fullrows:
        rr['candidate_id']=stable_hash(['original',rr['unit_id'],rr['han'],rr['viet'],rr['source_model']])
    migrated=migrate_legacy_judge_checkpoints(
        INPUT_ROOT, fullrows, SHARD, MODE, SIGNATURE, JUDGE_MODELS, stable_hash, utc_now)
    for key,rec in migrated.items():
        if key not in archive:archive[key]=rec
        if key in keys and key not in completed:completed[key]=rec
    print('Individually validated V2 long-prompt judgments imported:',len(migrated))
if archive:
    atomic_string(OUTPUT_DIR/'judge_checkpoint.jsonl',
        ''.join(json.dumps(rec,ensure_ascii=False)+'\n' for rec in archive.values()))
print('Merged archive (exact + prior FAST5 + V2):',len(archive))
print('OUTPUT_DIR:',OUTPUT_DIR)
print('Stored task keys:',len(TASKS),'| restored successes:',len(completed),
      '| total expected in all3 mode:',len(TASKS),
      '| baseline minimum in tiebreak mode:',len(rows)*2)
print('Restored from:',[str(p) for p in older])

# CHECKPOINT AUDIT — counts are for the selected TEST_MAX_UNITS, not archive.
from collections import Counter
scope_by_model=Counter()
provenance=Counter()
for cfg, row, key in TASKS:
    if key not in completed:
        continue
    rec=completed[key]
    scope_by_model[cfg['name']]+=1
    origin=rec.get('migrated_from_prompt') or 'same-signature'
    provenance[(cfg['name'],origin)]+=1
print('========== RESTORE AUDIT (CURRENT ROWS) ==========')
print('Rows:',len(rows),'| required base votes:',2*len(rows))
for cfg in JUDGE_MODELS:
    name=cfg['name']
    count=scope_by_model[name]
    print(' ',name,': restored',count,'/',len(rows), '| missing',len(rows)-count)
print('Restored provenance:')
for (name,origin),n in sorted(provenance.items()):
    print(' ',name,'|',origin,'|',n)
print('Total in-scope restored:',sum(scope_by_model.values()))
if JUDGE_EXECUTION_MODE=='tiebreak':
    names=[cfg['name'] for cfg in JUDGE_MODELS]
    keyed={(row['candidate_id'],cfg['name']): key for cfg,row,key in TASKS}
    pair_ready=0
    pair_needs_14b=0
    unresolved_14b=0
    already_done_14b=0
    for row in rows:
        k1=keyed[(row['candidate_id'],names[0])]
        k2=keyed[(row['candidate_id'],names[1])]
        k3=keyed[(row['candidate_id'],names[2])]
        if k1 in completed and k2 in completed:
            pair_ready+=1
            pair=arbitrate_two_judges(row,names[:2],
                {names[0]:completed[k1]['result'],names[1]:completed[k2]['result']},
                UNTRANSLATED_HAN_ALLOWLIST,JUDGE_SCORE_THRESHOLD)
            if pair['needs_arbiter']:
                pair_needs_14b+=1
                if k3 in completed:already_done_14b+=1
                else:unresolved_14b+=1
    print('Tiebreak: base pairs ready',pair_ready,'/',len(rows))
    print('Tiebreak: known cases requiring 14B',pair_needs_14b,
          '| 14B already cached',already_done_14b,
          '| 14B still needed',unresolved_14b)
    print('Tiebreak: base pairs missing',len(rows)-pair_ready,
          '(some may require 14B after completing 8B)')
print('==================================================')


In [ ]:
# CELL mới: đặt SAU Cell 6 + cell fast5_singlecase_hotfix V2, TRƯỚC Cell 7.
# Không thay đổi Cell 7 Deferred Retry hiện có.
from pathlib import Path
import sys

# Tìm đúng thư mục chứa các module V2/FAST5 đã import trong Cell 3.
assert 'RESOLVED_MODULE_DIR' in globals(), 'Chạy Cell 3 trước'
MODULE_DIR = Path(RESOLVED_MODULE_DIR)
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from fast5_feedback_retry import install as install_fast5_feedback
judge_batch_with_fallback = install_fast5_feedback(
    globals(), diagnostics_path=OUTPUT_DIR / 'judge_feedback_diagnostics.jsonl')

assert judge_batch_with_fallback.patch_version.startswith('fast5-targeted-feedback-')
print('FAST5 feedback patch: PASS')


## 7. Chạy Judge theo chiến lược all3 / tiebreak, batch 1–12 câu, JSON ngắn khi không có lỗi với tự chia nhỏ khi cần

`JUDGE_BATCH_SIZE=4` gộp tối đa bốn câu trong một lần sinh JSON; mỗi câu vẫn có **ngữ cảnh riêng, kết quả và checkpoint riêng**. Nếu JSON batch lỗi, vượt giới hạn token hoặc thiếu kết quả, hệ thống tự chia phần chưa đạt thành nhóm nhỏ hơn (đến từng câu nếu cần). Batch có thể nhanh hơn nhờ giảm prompt trùng lặp, **không bảo đảm giảm thời gian inference**, cần đo trên Kaggle.

**Lưu ý:** Hai phần tử `[ID, SCORE]` chỉ hợp lệ khi điểm ≥97, không có lỗi và không cần thêm ngữ cảnh. Lỗi thật, dù 97 điểm, vẫn phải trả danh sách lỗi đầy đủ.


In [ ]:
# CELL 7 — DEFERRED RETRY
import importlib
import fast5_deferred_retry

importlib.reload(fast5_deferred_retry)

RUN_RESULT = fast5_deferred_retry.run_judge_deferred(
    globals()
)

print("Judge run:", RUN_RESULT)

## 8. Xuất bảng điểm, lỗi và ứng viên Repair

In [ ]:
# CELL 8: BẢNG CHẤM ĐIỂM, CÁC LỖI, CÂU CẦN REPAIR (xuất cả khi chưa chấm xong).
base_cols=['source_shard','candidate_id','unit_id','han','viet','original_vi','source_line','section',
           'source_model','repair_model','mode','ctx_before_han','ctx_after_han']
summary_cols=base_cols+['status','median_score','judge_count','judge_expected','han_leftover_count',
          'score_conflict_votes','clean_judge_count','high_score_votes','raw_serious_claims',
          'screened_claim_count','consensus_issue_count','minority_high_risk_count',
          'triage_version','consensus_issues_json','review_issues_json','screened_issues_json',
          'individual_scores_json','feedback_json','missing_models',
          'decision_path','tiebreak_needed','tiebreak_done',
          'unverified_claim_count','unverified_claims_json']
summaries=[]
model_names=[m['name'] for m in JUDGE_MODELS]
keys_by_cid={(r['candidate_id'],m['name']):key for m,r,key in TASKS}
for row in rows:
    scores={}
    for name in model_names:
        k=keys_by_cid[(row['candidate_id'],name)]
        if k in completed:scores[name]=completed[k]['result']
    if JUDGE_EXECUTION_MODE=='all3':
        s=aggregate_judgments(row,model_names,scores,UNTRANSLATED_HAN_ALLOWLIST,JUDGE_SCORE_THRESHOLD)
        decision_path='all3'
        need_arbiter=False
    else:
        pair=arbitrate_two_judges(row,BASE_JUDGES,scores,
            UNTRANSLATED_HAN_ALLOWLIST,JUDGE_SCORE_THRESHOLD)
        need_arbiter=pair['needs_arbiter']
        if not pair['pair_ready']:
            s=pair['pair_summary'] or aggregate_judgments(row,BASE_JUDGES,scores,
                UNTRANSLATED_HAN_ALLOWLIST,JUDGE_SCORE_THRESHOLD)
            decision_path='pending_base'
        elif need_arbiter:
            if ARBITER_NAME in scores:
                s=aggregate_judgments(row,model_names,scores,
                    UNTRANSLATED_HAN_ALLOWLIST,JUDGE_SCORE_THRESHOLD)
                decision_path='resolved_by_14b'
            else:
                s=aggregate_judgments(row,model_names,scores,
                    UNTRANSLATED_HAN_ALLOWLIST,JUDGE_SCORE_THRESHOLD)
                decision_path='pending_14b'
        else:
            # Cơ chế cũ: 7B và 8B đồng thuận thì kết luận từ đúng hai phiếu.
            # Phiếu 14B đã lưu chỉ dùng khi chọn all3 hoặc thật sự cần phân xử.
            s=pair['pair_summary']
            decision_path=pair['reason']
            # Ngoại lệ an toàn: không tự PASS nếu checkpoint 14B có lỗi thiếu
            # bằng chứng. Không gọi model, chỉ yêu cầu review.
            if (ARBITER_NAME in scores and
                scores[ARBITER_NAME].get('requires_manual_review')):
                s=aggregate_judgments(row,model_names,scores,
                    UNTRANSLATED_HAN_ALLOWLIST,JUDGE_SCORE_THRESHOLD)
                decision_path='cached_14b_unverified_review'
    # Defensive compatibility: some earlier V2 pair summaries omit FAST5-only
    # diagnostic fields. Never fail an audit/export just because a model has
    # already returned a valid majority decision.
    s.setdefault('unverified_claim_count', 0)
    s.setdefault('unverified_claims_json', '[]')
    summaries.append({'source_shard':SHARD,**row,**s,
        'decision_path':decision_path,'tiebreak_needed':int(need_arbiter),
        'tiebreak_done':int(ARBITER_NAME in scores)})
dump_tsv(OUTPUT_DIR/'judge_summary.tsv',summaries,summary_cols)
repair_candidates=[r for r in summaries if r['status'] in ('REPAIR_REQUIRED','REVIEW_REQUIRED','CONTEXT_REVIEW')]
review_queue=[r for r in summaries if r['status'] in ('REVIEW_REQUIRED','CONTEXT_REVIEW')]
dump_tsv(OUTPUT_DIR/'review_queue.tsv',review_queue,summary_cols)
dump_tsv(OUTPUT_DIR/'repair_candidates.tsv',repair_candidates,summary_cols)
counts=Counter(r['status'] for r in summaries)
retry_attempts=Counter(str(rec.get('attempts_used',1)) for rec in completed.values())
format_fix_counts=Counter(fix for rec in completed.values()
                          for fix in rec.get('result',{}).get('format_fixes',[]))
output_tier_counts=Counter(str(rec.get('result',{}).get('fast_output_tier','legacy_or_unknown'))
                           for rec in completed.values())
stats={'schema':VERSION,'shard_id':SHARD,'stage':MODE,'signature':SIGNATURE,'source_sha256':SOURCE_SHA,
       'n_candidate_rows':len(rows),'score_threshold':JUDGE_SCORE_THRESHOLD,'total_inferences_expected':len(TASKS) if JUDGE_EXECUTION_MODE=='all3' else RUN_RESULT['expected_required_now'],
       'completed':len(completed),'execution_mode':JUDGE_EXECUTION_MODE,
       'tiebreak_needed':sum(r['tiebreak_needed'] for r in summaries),
       'tiebreak_done':sum(r['tiebreak_done'] and r['tiebreak_needed'] for r in summaries),
       'status_counts':dict(counts),'run':RUN_RESULT,'judge_models':model_names,
       'triage_version':TRIAGE_VERSION,
       'screened_claims_total':sum(r['screened_claim_count'] for r in summaries),
       'consensus_issues_total':sum(r['consensus_issue_count'] for r in summaries),
       'retry_attempt_histogram':dict(retry_attempts),'safe_format_fixes':dict(format_fix_counts),
       'output_tier_counts':dict(output_tier_counts),'short_output_min_score':SHORT_OUTPUT_MIN_SCORE,
       'unverified_claims_total':sum(r['unverified_claim_count'] for r in summaries),
       'mean_context_present':round(sum(bool(r.get('ctx_before_han') or r.get('ctx_after_han')) for r in rows)/len(rows),3),
       'note':'Score is subjective; short output means score>=97, errors=[], flags=0. No semantic correctness guarantee.'}
write_json(OUTPUT_DIR/'judge_report.json',stats)
print('Status:',dict(counts),'| completed:',len(completed),
      '| required in mode:',RUN_RESULT['expected_required_now'])
print('OUTPUT:',OUTPUT_DIR)
for f in ('judge_manifest.json','judge_checkpoint.jsonl','judge_summary.tsv',
          'repair_candidates.tsv','review_queue.tsv','judge_report.json'):
    p=OUTPUT_DIR/f;print('  ',f, p.stat().st_size if p.exists() else '(no file)')


## 9. Chạy lại / Stage tiếp theo

- **Từ Judge sang Repair:** Add Input **toàn bộ output** thư mục trong `yhct_judge_v2/<shard>/original/<signature>/` vào notebook Repair V2. File cần: `judge_manifest.json`, `repair_candidates.tsv`.
- **Từ Repair sang Re-judge:** Trong notebook **này**, đặt `MODE='rejudge'`, Add Input Repair output chứa `rejudge_inputs.tsv`. Context Hán trước/sau đi kèm từng candidate, không cần TSV gốc. Một câu có thể có nhiều bản Repair; mỗi bản có `candidate_id` độc lập.
- **Resume:** Add Input output phiên trước của **đúng Stage + Shard + signature**. Checkpoint từng inference có `fsync`. Nếu đổi prompt, Judge model, source hoặc context, các inference không còn khớp sẽ chạy lại.
- **Lưu ý:** `REPAIR_REQUIRED` là đồng thuận có *vùng lỗi trùng nhau* từ ≥2 Judge, **không phải chứng minh khách quan rằng lỗi có thật**. `REVIEW_REQUIRED`/`CONTEXT_REVIEW` cần rà soát trước khi thay văn bản chính. Chữ Hán còn sót được gắn cờ, kể cả score ≥95.
- Nếu một TSV nằm ở ranh giới phần, chỉ có context nội bộ; có thể Add Input **cả 4 TSV**, chọn `SOURCE_TSV_PATH` cho phần cần chấm để lấy câu lân cận. Không trộn câu L/P hoặc section khác.


**Phiên bản modular:** để chạy trên Kaggle, Add Input thêm dataset chứa 3 file `.py` V2. Khi sửa `.py` sau này, Restart Session rồi Run All; checkpoint chỉ nên dùng chung khi prompt/rubric và dữ liệu không thay đổi.

- **Voting mới:** `REPAIR_REQUIRED` chỉ khi đồng thuận chặt về category, span và bản sửa. `REVIEW_REQUIRED` có thể là phản biện lẻ nguy cơ cao. `ACCEPTED_WITH_NOTES` là chấp nhận tạm thời có ý kiến trái chiều, không phải ground truth. `review_queue.tsv` lưu riêng câu cần xem lại.

- **Batch Judge:** Nếu một lượt gọi 4–8 câu bị tràn token hoặc JSON lỗi, notebook tự chia đôi những câu chưa có kết quả hợp lệ cho đến khi chạy được. Dùng `JUDGE_BATCH_SIZE=1` nếu batch không ổn định hoặc không nhanh hơn.

- **Thử đủ ba model:** `JUDGE_EXECUTION_MODE='all3'`, `TEST_MAX_UNITS=80`. Sau khi chạy thử, có thể chuyển sang `tiebreak` và `TEST_MAX_UNITS=None`; kết quả đã chấm không phải làm lại nếu TSV/context/prompt không đổi.
- **Tăng batch:** mặc định 8/12/6, chỉ gộp prompt; dùng `judge_report.json` để so sánh tốc độ và số lần fallback. Nếu batch lớn liên tục lỗi, tự giảm cho model đó.
- **Tiebreak:** 7B + 8B mọi câu, chỉ gọi 14B với câu bất đồng; chấp nhận sơ bộ/Repair khi hai model đồng thuận. Chữ Hán còn sót được đưa Review mà không gọi 14B.

**Lưu ý FAST5:** Vì prompt thay đổi, điểm Judge V2 cũ không được tự khôi phục vào FAST5. Add Input output FAST5 phiên trước (nếu có). Không đặt kỳ vọng 5 giây trước khi benchmark GPU.

- **Cơ chế chấm khôi phục:** chấp nhận sơ bộ theo đa số phiếu từ ngưỡng >=95 cùng với kiểm tra lỗi và rủi ro của `yhct_judge_v2.aggregate_judgments`; không ép `REVIEW_REQUIRED` chỉ vì hai/ba điểm số lệch >=25. Trong `tiebreak`, Judge 14B chỉ chạy khi `yhct_judge_v2.arbitrate_two_judges` thấy cần phân xử.
- **Giữ checkpoint:** FAST5 prompt, `FAST_JUDGE_VERSION` và chữ ký nội dung trong Cell 6 không thay đổi. Đổi chính sách hậu xử lý không giải quyết được việc `Restored from: []` nếu FAST5 output cũ mang signature khác do *phiên bản prompt* thay đổi.
